# QLoRA Evaluation on Test Data

This section evaluates the trained QLoRA adapter on the representative
test dataset.

The evaluation uses the same test examples and evaluation pipeline used
for the baseline and LoRA experiments.

Metrics collected:

- Exact match
- Tool-name accuracy
- Argument accuracy
- Call-count accuracy
- Complexity-wise performance
- Total evaluation time
- Average inference time
- Model loading time
- System resource metrics

All evaluation results, metrics, and artifacts are tracked using MLflow.

In [ ]:
!pip install -q \
    transformers==5.16.1 \
    datasets==5.0.1 \
    accelerate \
    peft==0.20.0 \
    trl==1.12.0 \
    bitsandbytes==0.50.2 \
    mlflow==3.16.0 \
    huggingface_hub \
    pyyaml \
    pandas \
    numpy \
    psutil

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

print("Google Drive mounted.")

In [ ]:
import json
import time
import os

import torch
import mlflow
import pandas as pd

from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

print("Imports successful.")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Experiment Configuration

The paths below point directly to the files stored in Google Drive.

The QLoRA adapter is loaded from the checkpoint produced during training.

The representative test dataset contains 1,000 examples and is the same
dataset used for the other model evaluations.

In [ ]:
PROJECT_ROOT = "/content/drive/MyDrive/tool_calling_project"

TEST_DATA_PATH = (
    f"{PROJECT_ROOT}/data/"
    "representative_data/test"
)

QLORA_ADAPTER_PATH = (
    f"{PROJECT_ROOT}/checkpoints/"
    "qlora_representative_4000_final"
)

RESULTS_DIR = (
    f"{PROJECT_ROOT}/results"
)

QLORA_RESULTS_FILE = (
    f"{RESULTS_DIR}/qlora_results_1000.csv"
)

QLORA_COMPLEXITY_FILE = (
    f"{RESULTS_DIR}/qlora_complexity_1000.csv"
)

QLORA_METRICS_FILE = (
    f"{RESULTS_DIR}/qlora_metrics_1000.json"
)

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

MAX_NEW_TOKENS = 256

QLORA_NUM_SAMPLES = 1000

MLFLOW_DB_PATH = (
    f"{PROJECT_ROOT}/mlflow.db"
)

MLFLOW_TRACKING_URI = (
    f"sqlite:///{MLFLOW_DB_PATH}"
)

MLFLOW_EXPERIMENT_NAME = (
    "tool_calling_finetuning"
)

os.makedirs(
    RESULTS_DIR,
    exist_ok=True
)

print("Configuration loaded.")

## Load Representative Test Dataset

Load the persisted representative test dataset from Google Drive.

The dataset is intentionally kept identical across baseline, LoRA,
and QLoRA evaluation.

In [ ]:
test_dataset = load_from_disk(
    TEST_DATA_PATH
)

print(
    f"Test dataset size: "
    f"{len(test_dataset)}"
)

print(
    f"Columns: "
    f"{test_dataset.column_names}"
)

## Load LORA/QLoRA Model

The trained QLoRA adapter is loaded on top of the base Qwen model.

The model-loading approach remains the same as the existing evaluation
pipeline:

1. Load the base Qwen model.
2. Load the QLoRA adapter using PEFT.
3. Switch the model to evaluation mode.

QLoRA training used 4-bit quantization, but the evaluation code does not
need to modify the adapter itself.

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

from peft import PeftModel

import torch
import time


bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)


print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)


print("Loading 4-bit QLoRA base model...")

model_loading_start = time.perf_counter()

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)


print("Loading QLoRA adapter...")

model = PeftModel.from_pretrained(
    model,
    QLORA_ADAPTER_PATH
)


model.eval()


model_loading_time_seconds = (
    time.perf_counter()
    - model_loading_start
)


print("\nQLoRA model loaded successfully.")
print(
    f"Model loading time: "
    f"{model_loading_time_seconds:.2f} seconds"
)

In [ ]:
trainable_params = 0
total_params = 0

for parameter in model.parameters():

    total_params += parameter.numel()

    if parameter.requires_grad:
        trainable_params += parameter.numel()


trainable_percentage = (
    100
    * trainable_params
    / total_params
)


print(
    f"Total parameters: "
    f"{total_params:,}"
)

print(
    f"Trainable parameters: "
    f"{trainable_params:,}"
)

print(
    f"Trainable percentage: "
    f"{trainable_percentage:.4f}%"
)

## Tool-Call Inference

This function generates the model response for one example.

The model receives the formatted tool-calling prompt and generates the
assistant response containing the predicted tool calls.

In [ ]:
def predict_tool_calls(
    example,
    model,
    tokenizer,
    max_new_tokens=256
):

    prompt = example["prompt"]

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[
        0
    ][
        inputs["input_ids"].shape[1]:
    ]

    raw_response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return raw_response

## Parse Tool Calls

The model output is expected to contain one or more tool-call blocks.

Each tool call follows the structure:

`<tool_call>{"name": ..., "arguments": ...}</tool_call>`

The parser extracts the JSON representation of each predicted tool call.

In [ ]:
import re


def parse_tool_calls(
    response
):

    pattern = (
        r"<tool_call>\s*"
        r"(.*?)"
        r"\s*</tool_call>"
    )

    matches = re.findall(
        pattern,
        response,
        flags=re.DOTALL
    )

    tool_calls = []

    for match in matches:

        try:

            call = json.loads(
                match.strip()
            )

            tool_calls.append(
                call
            )

        except json.JSONDecodeError:

            continue

    return tool_calls

In [ ]:
## Evaluation Metrics

For every example, compare the expected tool calls with the predicted
tool calls.

The evaluation calculates:

- Tool-name correctness
- Argument correctness
- Call-count correctness
- Exact match

These per-example results are later aggregated across the full test set.

In [ ]:
def normalize_calls(calls):

    normalized = []

    for call in calls:

        normalized.append({
            "name": call.get(
                "name"
            ),
            "arguments": call.get(
                "arguments",
                {}
            )
        })

    return normalized


def evaluate_prediction(
    expected_calls,
    predicted_calls
):

    expected_calls = normalize_calls(
        expected_calls
    )

    predicted_calls = normalize_calls(
        predicted_calls
    )


    call_count_match = (
        len(expected_calls)
        == len(predicted_calls)
    )


    if len(expected_calls) == 0:

        tool_name_match = (
            len(predicted_calls) == 0
        )

        arguments_match = (
            len(predicted_calls) == 0
        )

    else:

        tool_name_match = (
            len(expected_calls)
            == len(predicted_calls)
            and all(
                expected["name"]
                == predicted["name"]

                for expected, predicted
                in zip(
                    expected_calls,
                    predicted_calls
                )
            )
        )


        arguments_match = (
            len(expected_calls)
            == len(predicted_calls)
            and all(
                expected["arguments"]
                == predicted["arguments"]

                for expected, predicted
                in zip(
                    expected_calls,
                    predicted_calls
                )
            )
        )


    exact_match = (
        expected_calls
        == predicted_calls
    )


    return {

        "exact_match":
            int(exact_match),

        "tool_name_match":
            int(tool_name_match),

        "arguments_match":
            int(arguments_match),

        "call_count_match":
            int(call_count_match),
    }

In [ ]:
def calculate_overall_metrics(
    evaluation_results
):

    total = len(
        evaluation_results
    )

    if total == 0:

        return {
            "total_examples": 0,
            "exact_match": 0.0,
            "tool_name_match": 0.0,
            "arguments_match": 0.0,
            "call_count_match": 0.0,
        }


    return {

        "total_examples":
            total,

        "exact_match":
            sum(
                r["exact_match"]
                for r in evaluation_results
            ) / total,

        "tool_name_match":
            sum(
                r["tool_name_match"]
                for r in evaluation_results
            ) / total,

        "arguments_match":
            sum(
                r["arguments_match"]
                for r in evaluation_results
            ) / total,

        "call_count_match":
            sum(
                r["call_count_match"]
                for r in evaluation_results
            ) / total,
    }

## Run Evaluation and Track with MLflow

This is the main evaluation cell.

For each test example:

1. Generate a response.
2. Parse predicted tool calls.
3. Load expected tool calls.
4. Calculate evaluation metrics.
5. Store the detailed result.

After inference, overall metrics, complexity metrics, timing information,
and evaluation artifacts are logged to MLflow.

In [ ]:
import os
import json
import time
import threading
from pathlib import Path

import pandas as pd
import psutil
import torch
import mlflow

from tqdm.auto import tqdm


# ============================================================
# CONFIGURATION
# ============================================================

NUM_SAMPLES = 1000
CHECKPOINT_INTERVAL = 50

RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_FILE = RESULTS_DIR / "qlora_results_1000.csv"
METRICS_FILE = RESULTS_DIR / "qlora_metrics_1000.json"
COMPLEXITY_FILE = RESULTS_DIR / "qlora_complexity_1000.csv"

CHECKPOINT_FILE = RESULTS_DIR / "qlora_eval_checkpoint.csv"

MLFLOW_TRACKING_URI = f"sqlite:///{(PROJECT_ROOT / 'mlflow.db').as_posix()}"
MLFLOW_EXPERIMENT_NAME = "tool_calling_finetuning"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)


# ============================================================
# MONITORING
# ============================================================

monitoring = {
    "cpu_percent": [],
    "ram_percent": [],
    "gpu_utilization_percent": [],
    "gpu_memory_used_mb": [],
    "gpu_memory_reserved_mb": []
}

monitoring_stop = threading.Event()


def monitor_resources():
    while not monitoring_stop.is_set():

        # CPU / RAM
        monitoring["cpu_percent"].append(psutil.cpu_percent(interval=None))
        monitoring["ram_percent"].append(psutil.virtual_memory().percent)

        # GPU
        if torch.cuda.is_available():
            try:
                device = torch.cuda.current_device()

                memory_used = torch.cuda.memory_allocated(device) / (1024 ** 2)
                memory_reserved = torch.cuda.memory_reserved(device) / (1024 ** 2)

                monitoring["gpu_memory_used_mb"].append(memory_used)
                monitoring["gpu_memory_reserved_mb"].append(memory_reserved)

                # NVML GPU utilization
                try:
                    import pynvml

                    pynvml.nvmlInit()
                    handle = pynvml.nvmlDeviceGetHandleByIndex(device)
                    utilization = pynvml.nvmlDeviceGetUtilizationRates(handle)

                    monitoring["gpu_utilization_percent"].append(
                        utilization.gpu
                    )

                    pynvml.nvmlShutdown()

                except Exception:
                    pass

            except Exception:
                pass

        time.sleep(1)


# ============================================================
# METRIC HELPERS
# ============================================================

def normalize_calls(value):
    """
    Convert expected answers or model output into a normalized
    list of tool-call dictionaries.
    """

    if value is None:
        return []

    if isinstance(value, list):
        return value

    if isinstance(value, dict):
        return [value]

    if not isinstance(value, str):
        return []

    text = value.strip()

    # Remove chat termination token
    text = text.replace("<|im_end|>", "").strip()

    # Extract <tool_call> blocks
    if "<tool_call>" in text:
        blocks = text.split("<tool_call>")[1:]

        calls = []

        for block in blocks:
            if "</tool_call>" in block:
                block = block.split("</tool_call>", 1)[0]

            block = block.strip()

            if not block:
                continue

            try:
                calls.append(json.loads(block))
            except json.JSONDecodeError:
                continue

        if calls:
            return calls

    # Try direct JSON
    try:
        parsed = json.loads(text)

        if isinstance(parsed, list):
            return parsed

        if isinstance(parsed, dict):
            return [parsed]

    except json.JSONDecodeError:
        pass

    return []


def canonicalize_calls(calls):
    """
    Canonical representation used for exact-match comparison.
    """

    canonical = []

    for call in calls:

        if not isinstance(call, dict):
            continue

        name = call.get("name", call.get("tool_name"))

        arguments = call.get(
            "arguments",
            call.get("parameters", {})
        )

        canonical.append(
            {
                "name": name,
                "arguments": arguments
            }
        )

    return canonical


def tool_name_match(expected, predicted):

    expected = canonicalize_calls(expected)
    predicted = canonicalize_calls(predicted)

    if len(expected) != len(predicted):
        return False

    return all(
        e["name"] == p["name"]
        for e, p in zip(expected, predicted)
    )


def argument_match(expected, predicted):

    expected = canonicalize_calls(expected)
    predicted = canonicalize_calls(predicted)

    if len(expected) != len(predicted):
        return False

    return all(
        e["arguments"] == p["arguments"]
        for e, p in zip(expected, predicted)
    )


def exact_match(expected, predicted):

    expected = canonicalize_calls(expected)
    predicted = canonicalize_calls(predicted)

    return expected == predicted


def call_count_match(expected, predicted):

    return len(expected) == len(predicted)


def word_accuracy(expected, predicted):

    """
    Token/word-level accuracy over the canonical JSON
    representation.

    This measures how closely the generated tool-call text
    matches the expected tool-call representation.
    """

    expected_text = json.dumps(
        canonicalize_calls(expected),
        sort_keys=True
    )

    predicted_text = json.dumps(
        canonicalize_calls(predicted),
        sort_keys=True
    )

    expected_words = expected_text.split()
    predicted_words = predicted_text.split()

    if not expected_words:
        return 1.0 if not predicted_words else 0.0

    correct = sum(
        e == p
        for e, p in zip(expected_words, predicted_words)
    )

    return correct / max(
        len(expected_words),
        len(predicted_words)
    )


# ============================================================
# GENERATION
# ============================================================

def generate_prediction(example):

    tools = (
        json.loads(example["tools"])
        if isinstance(example["tools"], str)
        else example["tools"]
    )

    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful assistant with access to the following tools.\n\n"
                + json.dumps(tools, indent=2)
            )
        },
        {
            "role": "user",
            "content": example["query"]
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start = time.perf_counter()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    inference_time = time.perf_counter() - start

    generated_tokens = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    prediction = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=False
    )

    return prediction, inference_time


# ============================================================
# RESUME FROM CHECKPOINT IF AVAILABLE
# ============================================================

if CHECKPOINT_FILE.exists():

    results_df = pd.read_csv(CHECKPOINT_FILE)

    completed_ids = set(results_df["id"].astype(str))

    print(
        f"Found checkpoint with {len(results_df)} completed examples."
    )

else:

    results_df = pd.DataFrame()

    completed_ids = set()

    print("No previous checkpoint found. Starting evaluation.")


evaluation_examples = test_dataset.select(
    range(min(NUM_SAMPLES, len(test_dataset)))
)

remaining_examples = [
    example
    for example in evaluation_examples
    if str(example["id"]) not in completed_ids
]

print(f"Total examples: {len(evaluation_examples)}")
print(f"Remaining examples: {len(remaining_examples)}")


# ============================================================
# START MONITORING
# ============================================================

monitor_thread = threading.Thread(
    target=monitor_resources,
    daemon=True
)

monitor_thread.start()


# ============================================================
# MLflow RUN
# ============================================================

with mlflow.start_run(
    run_name="qlora_representative_1000_eval"
) as run:

    mlflow.log_params(
        {
            "model": MODEL_NAME,
            "adapter": str(QLORA_ADAPTER_PATH),
            "num_samples": NUM_SAMPLES,
            "quantization": "4-bit NF4",
            "double_quantization": True,
            "compute_dtype": "float16",
            "max_new_tokens": 256,
            "do_sample": False,
            "checkpoint_interval": CHECKPOINT_INTERVAL,
            "evaluation_dataset": "representative_qlora_test",
        }
    )

    evaluation_start = time.perf_counter()

    for example in tqdm(
        remaining_examples,
        desc="Evaluating QLoRA"
    ):

        prediction, inference_time = generate_prediction(example)

        expected_calls = normalize_calls(
            example["answers"]
        )

        predicted_calls = normalize_calls(
            prediction
        )

        row = {
            "id": example["id"],
            "query": example["query"],
            "expected": json.dumps(
                expected_calls,
                ensure_ascii=False
            ),
            "prediction": prediction,
            "predicted_calls": json.dumps(
                predicted_calls,
                ensure_ascii=False
            ),
            "num_calls": example["num_calls"],
            "complexity": example["complexity"],
            "inference_time_seconds": inference_time,

            "exact_match": int(
                exact_match(
                    expected_calls,
                    predicted_calls
                )
            ),

            "tool_name_match": int(
                tool_name_match(
                    expected_calls,
                    predicted_calls
                )
            ),

            "argument_match": int(
                argument_match(
                    expected_calls,
                    predicted_calls
                )
            ),

            "call_count_match": int(
                call_count_match(
                    expected_calls,
                    predicted_calls
                )
            ),

            "word_accuracy": word_accuracy(
                expected_calls,
                predicted_calls
            )
        }

        results_df = pd.concat(
            [
                results_df,
                pd.DataFrame([row])
            ],
            ignore_index=True
        )

        # ----------------------------------------------------
        # CHECKPOINT
        # ----------------------------------------------------

        if len(results_df) % CHECKPOINT_INTERVAL == 0:

            results_df.to_csv(
                CHECKPOINT_FILE,
                index=False
            )

            print(
                f"Checkpoint saved: {len(results_df)} examples"
            )

    evaluation_time = (
        time.perf_counter() - evaluation_start
    )


# ============================================================
# STOP MONITORING
# ============================================================

monitoring_stop.set()
monitor_thread.join(timeout=2)


# ============================================================
# FINAL SAVE
# ============================================================

results_df.to_csv(
    RESULTS_FILE,
    index=False
)

print(f"\nResults saved to: {RESULTS_FILE}")


# ============================================================
# OVERALL METRICS
# ============================================================

overall_metrics = {

    "num_samples": int(len(results_df)),

    "exact_match_accuracy": float(
        results_df["exact_match"].mean()
    ),

    "tool_name_accuracy": float(
        results_df["tool_name_match"].mean()
    ),

    "argument_accuracy": float(
        results_df["argument_match"].mean()
    ),

    "call_count_accuracy": float(
        results_df["call_count_match"].mean()
    ),

    "word_accuracy": float(
        results_df["word_accuracy"].mean()
    ),

    "average_inference_time_seconds": float(
        results_df["inference_time_seconds"].mean()
    ),

    "total_inference_time_seconds": float(
        results_df["inference_time_seconds"].sum()
    ),

    "evaluation_wall_time_seconds": float(
        evaluation_time
    )
}


# ============================================================
# COMPLEXITY METRICS
# ============================================================

complexity_metrics = (
    results_df
    .groupby("complexity")
    .agg(
        samples=("id", "count"),
        exact_match_accuracy=("exact_match", "mean"),
        tool_name_accuracy=("tool_name_match", "mean"),
        argument_accuracy=("argument_match", "mean"),
        call_count_accuracy=("call_count_match", "mean"),
        word_accuracy=("word_accuracy", "mean"),
        average_inference_time_seconds=(
            "inference_time_seconds",
            "mean"
        )
    )
    .reset_index()
)

complexity_metrics.to_csv(
    COMPLEXITY_FILE,
    index=False
)


# ============================================================
# RESOURCE METRICS
# ============================================================

def safe_mean(values):

    return (
        float(sum(values) / len(values))
        if values
        else None
    )


def safe_max(values):

    return (
        float(max(values))
        if values
        else None
    )


resource_metrics = {

    "average_cpu_percent": safe_mean(
        monitoring["cpu_percent"]
    ),

    "max_cpu_percent": safe_max(
        monitoring["cpu_percent"]
    ),

    "average_ram_percent": safe_mean(
        monitoring["ram_percent"]
    ),

    "max_ram_percent": safe_max(
        monitoring["ram_percent"]
    ),

    "average_gpu_utilization_percent": safe_mean(
        monitoring["gpu_utilization_percent"]
    ),

    "max_gpu_utilization_percent": safe_max(
        monitoring["gpu_utilization_percent"]
    ),

    "average_gpu_memory_used_mb": safe_mean(
        monitoring["gpu_memory_used_mb"]
    ),

    "max_gpu_memory_used_mb": safe_max(
        monitoring["gpu_memory_used_mb"]
    ),

    "average_gpu_memory_reserved_mb": safe_mean(
        monitoring["gpu_memory_reserved_mb"]
    ),

    "max_gpu_memory_reserved_mb": safe_max(
        monitoring["gpu_memory_reserved_mb"]
    )
}


# ============================================================
# COMBINE METRICS
# ============================================================

final_metrics = {
    **overall_metrics,
    **resource_metrics
}


with open(
    METRICS_FILE,
    "w"
) as f:

    json.dump(
        final_metrics,
        f,
        indent=2
    )


# ============================================================
# LOG TO MLFLOW
# ============================================================

mlflow.log_metrics(
    {
        key: value
        for key, value in final_metrics.items()
        if value is not None
    }
)

mlflow.log_artifact(
    str(RESULTS_FILE)
)

mlflow.log_artifact(
    str(COMPLEXITY_FILE)
)

mlflow.log_artifact(
    str(METRICS_FILE)
)


# ============================================================
# COMPLEXITY METRICS → MLFLOW
# ============================================================

for _, row in complexity_metrics.iterrows():

    complexity = row["complexity"]

    mlflow.log_metrics(
        {
            f"{complexity}_exact_match_accuracy":
                float(row["exact_match_accuracy"]),

            f"{complexity}_tool_name_accuracy":
                float(row["tool_name_accuracy"]),

            f"{complexity}_argument_accuracy":
                float(row["argument_accuracy"]),

            f"{complexity}_call_count_accuracy":
                float(row["call_count_accuracy"]),

            f"{complexity}_word_accuracy":
                float(row["word_accuracy"]),

            f"{complexity}_average_inference_time":
                float(row["average_inference_time_seconds"])
        }
    )


# ============================================================
# FINAL OUTPUT
# ============================================================

print("\n" + "=" * 60)
print("QLoRA EVALUATION COMPLETE")
print("=" * 60)

print("\nOverall metrics:")

for key, value in overall_metrics.items():

    print(f"{key}: {value}")

print("\nResource metrics:")

for key, value in resource_metrics.items():

    print(f"{key}: {value}")

print("\nComplexity breakdown:")
display(complexity_metrics)

print("\nMLflow run ID:")
print(run.info.run_id)